# 08. Ajuste de parâmetros das técnicas clássicas

Este notebook ajusta apenas parâmetros das técnicas já existentes (ELA, JPEG Ghost e busca cega de copy-move), usando um conjunto de desenvolvimento separado. O classificador, o gerador sintético e a faixa de qualidades do JPEG Ghost (50 a 90) não são alterados.

A medida é a **localização da caixa colada**: em cada nota adulterada, a técnica acerta ou não a caixa registrada, e o acerto é comparado ao de pseudo-regiões sorteadas ao acaso (mesmo tamanho, com texto, sem sobreposição com a região anotada), pelo protocolo `evaluate_region_technique` com seed 42. Isso mede localização em colagens sintéticas exatas, não detecção de fraudes reais.

Os números aparecem apenas como saída das células; este notebook não registra conclusões.

In [1]:
import sys
sys.path.append("../src")
import warnings
warnings.filterwarnings("ignore", message=".*encountered in matmul")

import hashlib
import json
from pathlib import Path

import pandas as pd

from ajuste_parametros import PADROES, avaliar, avaliar_grade, calcular_pares, escolher
from avaliacao_sintetica import carregar_amostras

RAIZ = Path("..")

## 1. Grade e critério congelados

A grade, o critério de escolha e o hash do código de ajuste foram gravados em `results/grade_ajuste.json` antes de qualquer execução. A célula confere que o código atual e o gerador ainda têm os hashes registrados.

Parâmetros avaliados: ELA, qualidade de recompressão em {80, 90} e percentil do erro usado como score da região em {95, 99}; JPEG Ghost, bloco em {8, 16, 32}; busca cega, limiar de correlação em {0,95; 0,979; 0,99} e cobertura mínima da região em {0,3; 0,5}. As configurações padrão são ELA q=90 com média do erro (medida do notebook 01), Ghost 16 e busca cega com limiar 0,99 e cobertura 0,5.

Critério: cada técnica é escolhida separadamente; uma configuração só substitui a padrão se a diferença (acerto real menos acaso) crescer pelo menos 5 pontos percentuais **e** o intervalo de Wilson de 95% do acerto real não se sobrepuser ao do acaso. Caso nenhuma cumpra, mantém-se a padrão.

In [2]:
grade = json.loads((RAIZ / "results/grade_ajuste.json").read_text(encoding="utf-8"))
gerador = json.loads((RAIZ / "results/gerador_congelado.json").read_text(encoding="utf-8"))
sha = lambda caminho: hashlib.sha256((RAIZ / caminho).read_bytes()).hexdigest()
print("congelado em:", grade["congelado_em"])
print("código de ajuste idêntico ao congelado:", sha(grade["arquivo"]) == grade["sha256"])
print("gerador idêntico ao congelado:", sha(gerador["arquivo"]) == gerador["sha256"] == grade["gerador_sha256"])
print(json.dumps(grade["grade"], ensure_ascii=False))

congelado em: 2026-11-03T12:59:39
código de ajuste idêntico ao congelado: True
gerador idêntico ao congelado: True
{"ela": {"qualidade": [80, 90], "percentil": [95, 99]}, "ghost": {"block_size": [8, 16, 32], "qualidades": "50 a 90 passo 5 (fixa)"}, "copy_move_busca_cega": {"limiar": [0.95, 0.979, 0.99], "cobertura": [0.3, 0.5]}, "classificador": "não alterado"}


## 2. Conjunto de desenvolvimento

As 800 notas do split de treino do CORD nunca foram usadas no projeto. O gerador congelado (seed 42) foi aplicado a cada uma, e as colagens bem-sucedidas formam `data/cord_sintetico_dev/`. Como a medida é de localização, só as notas adulteradas são necessárias. Notas em que o gerador não acha par origem/destino ficam de fora.

In [3]:
dev = carregar_amostras(RAIZ / "data/cord_sintetico_dev")
print("notas adulteradas de desenvolvimento:", len(dev), "| notas do treino sem par (excluídas):", grade["desenvolvimento"]["notas_sem_par_excluidas"])

notas adulteradas de desenvolvimento: 766 | notas do treino sem par (excluídas): 34


## 3. Grade no desenvolvimento

Cada configuração é avaliada no conjunto de desenvolvimento (uma por processo). Os pares da busca cega são calculados uma vez por imagem, com piso de correlação 0,95, e os limiares maiores apenas filtram esses pares. A tabela completa vai para `results/ajuste_desenvolvimento.csv`.

In [4]:
tabela = pd.DataFrame(avaliar_grade(dev))
tabela.to_csv(RAIZ / "results/ajuste_desenvolvimento.csv", index=False)
pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 30)
print(tabela.drop(columns=["exclusoes"]).round(3).to_string(index=False))
print(tabela[["tecnica", "exclusoes"]].drop_duplicates().to_string(index=False))

  tecnica  qualidade  percentil   n  acertos  taxa_real  real_ic_min  real_ic_max  taxa_acaso  acaso_ic_min  acaso_ic_max  diferenca  block_size  limiar  cobertura
      ela       90.0        NaN 766      690      0.901        0.878        0.920       0.445         0.438         0.453      0.455         NaN     NaN        NaN
      ela       80.0       95.0 766      664      0.867        0.841        0.889       0.359         0.352         0.367      0.508         NaN     NaN        NaN
      ela       80.0       99.0 766      673      0.879        0.854        0.900       0.414         0.407         0.422      0.464         NaN     NaN        NaN
      ela       90.0       95.0 766      655      0.855        0.828        0.878       0.292         0.285         0.300      0.563         NaN     NaN        NaN
      ela       90.0       99.0 766      643      0.839        0.812        0.864       0.336         0.329         0.343      0.503         NaN     NaN        NaN
    ghost       

## 4. Escolha pelo critério congelado

Para cada técnica, aplica-se o critério da seção 1 às linhas da tabela.

In [5]:
escolhas = {}
for tecnica, padrao in PADROES.items():
    linhas = tabela[tabela["tecnica"] == tecnica].astype(object).where(lambda t: t.notna(), None).to_dict("records")
    escolhida, trocou = escolher(linhas, padrao)
    parametros = {k: (int(escolhida[k]) if k in ("qualidade", "block_size") else escolhida[k]) for k in padrao}
    escolhas[tecnica] = {"parametros": parametros, "trocou": trocou}
    print(tecnica, "->", escolhas[tecnica], "| diferença escolhida: %.3f" % escolhida["diferenca"])

ela -> {'parametros': {'qualidade': 90, 'percentil': 95.0}, 'trocou': True} | diferença escolhida: 0.563
ghost -> {'parametros': {'block_size': 16}, 'trocou': False} | diferença escolhida: 0.032
copy_move -> {'parametros': {'limiar': 0.99, 'cobertura': 0.5}, 'trocou': False} | diferença escolhida: 0.018


## 5. Avaliação final (segundo uso do conjunto de validação e teste)

A configuração padrão e a escolhida são avaliadas, uma única vez, nas 95 notas adulteradas de `data/cord_sintetico/` (validation e test do CORD). **Este é o segundo uso desse conjunto:** o primeiro foi a avaliação do notebook 05. Por isso o resultado não é independente de uma decisão tomada olhando o conjunto anterior, embora a escolha dos parâmetros tenha usado só o desenvolvimento. Resultados em `results/ajuste_final.json`.

In [6]:
final_amostras = [s for s in carregar_amostras(RAIZ / "data/cord_sintetico") if s.is_tampered]
pares_final = calcular_pares(final_amostras)
final = {"aviso": "segundo uso de data/cord_sintetico/ (o primeiro foi o notebook 05)", "n_adulteradas": len(final_amostras)}
for tecnica, padrao in PADROES.items():
    escolhido = escolhas[tecnica]["parametros"]
    final[tecnica] = {"padrao": avaliar(tecnica, padrao, final_amostras, pares_final),
                      "escolhida": avaliar(tecnica, escolhido, final_amostras, pares_final),
                      "trocou": escolhas[tecnica]["trocou"]}
    for nome in ("padrao", "escolhida"):
        l = final[tecnica][nome]
        print(f"{tecnica:10} {nome:9} n={l['n']} acerto={l['taxa_real']:.1%} [{l['real_ic_min']:.1%}; {l['real_ic_max']:.1%}] "
              f"acaso={l['taxa_acaso']:.1%} [{l['acaso_ic_min']:.1%}; {l['acaso_ic_max']:.1%}] {l['exclusoes']}")
(RAIZ / "results/ajuste_final.json").write_text(json.dumps(final, ensure_ascii=False, indent=2, default=float), encoding="utf-8")

ela        padrao    n=95 acerto=89.5% [81.7%; 94.2%] acaso=45.0% [42.8%; 47.2%] {'text_mask_failed': 0, 'real_region_failed': 0, 'few_valid_pseudo': 0}
ela        escolhida n=95 acerto=87.4% [79.2%; 92.6%] acaso=31.2% [29.1%; 33.3%] {'text_mask_failed': 0, 'real_region_failed': 0, 'few_valid_pseudo': 0}


ghost      padrao    n=21 acerto=14.3% [5.0%; 34.6%] acaso=1.0% [0.4%; 3.0%] {'text_mask_failed': 0, 'real_region_failed': 33, 'few_valid_pseudo': 41}
ghost      escolhida n=21 acerto=14.3% [5.0%; 34.6%] acaso=1.0% [0.4%; 3.0%] {'text_mask_failed': 0, 'real_region_failed': 33, 'few_valid_pseudo': 41}


copy_move  padrao    n=95 acerto=1.1% [0.2%; 5.7%] acaso=0.0% [-0.0%; 0.2%] {'text_mask_failed': 0, 'real_region_failed': 0, 'few_valid_pseudo': 0}
copy_move  escolhida n=95 acerto=1.1% [0.2%; 5.7%] acaso=0.0% [-0.0%; 0.2%] {'text_mask_failed': 0, 'real_region_failed': 0, 'few_valid_pseudo': 0}


3245